In [ ]:
# ===== NB15 | CELL 1: header =====
from google.colab import drive
drive.mount('/content/drive')
print('CELL 1 DONE')

In [ ]:
# ===== NB15 | CELL 2: error analysis on the internal test set =====
# WHAT: count false negatives / false positives per label, measure errors by comment length, and save a sample
#       of errors to tag by hand (van Aken et al. did the same with 200 samples per error type).
# WHY:  scores say HOW OFTEN the model fails. Reading the errors says WHY (label noise, no swear words,
#       sarcasm, quotes, truncation), and that is the research contribution of the paper.
import sys, importlib, json, os, re
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, load_split
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
for d in ['graphs', 'tables']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json')); thr = cfg['thresholds']
df = load_split(PROJECT_DIR, 'test_internal')
Pf = np.load(f'{PROJECT_DIR}/results/final/final_test_internal.npy')
Y = df[LABELS].values
flag = Pf >= np.array([thr[l] for l in LABELS])
df['words'] = df.comment_text.fillna('').str.split().str.len()

# T9a: how many false negatives / false positives per label
rows = []
for i, l in enumerate(LABELS):
    rows.append({'label': l, 'positives': int(Y[:, i].sum()), 'false_negatives': int(((Y[:, i] == 1) & ~flag[:, i]).sum()),
                 'false_positives': int(((Y[:, i] == 0) & flag[:, i]).sum())})
E = pd.DataFrame(rows); print('Error counts on the internal test set:'); print(E.to_string(index=False))
E.to_csv(f'{PROJECT_DIR}/results/tables/T9a_error_counts.csv', index=False)

# F15: error rate by comment length (toxic label). 100 words is roughly 128 tokens.
bins = [0, 20, 50, 100, 200, 100000]; labels_ = ['<=20', '21-50', '51-100', '101-200', '>200']
df['bin'] = pd.cut(df.words, bins=bins, labels=labels_)
y, f = Y[:, 0] == 1, flag[:, 0]
fn_rate = df[y].assign(err=~f[y]).groupby('bin', observed=True).err.mean()
fp_rate = df[~y].assign(err=f[~y]).groupby('bin', observed=True).err.mean()
cnt = df.groupby('bin', observed=True).size()
print('\nToxic label, error rate by comment length (words):')
print(pd.DataFrame({'comments': cnt, 'FN rate (of toxic)': fn_rate.round(3), 'FP rate (of clean)': fp_rate.round(3)}))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
fn_rate.plot(kind='bar', ax=ax[0], color='#D9776B', title='missed toxic comments (FN rate)'); fp_rate.plot(kind='bar', ax=ax[1], color='#4C72B0', title='false alarms (FP rate)')
plt.suptitle('F15: error rate by comment length'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F15_error_by_length.png', dpi=150); plt.show()
print('Share of toxic comments longer than 100 words:', round(float((df[y].words > 100).mean()), 3))

# sample errors for manual tagging, with simple automatic hints
SWEAR = r'\b(?:fuck\w*|shit\w*|damn\w*|ass|asshole\w*|bitch\w*|crap|stupid|idiot\w*|moron\w*|dick\w*|bastard\w*|cunt\w*|nigg\w*|fag\w*)\b'
plan = {'toxic': 100, 'threat': 30, 'identity_hate': 30}
out = []
for l, k in plan.items():
    i = LABELS.index(l)
    for et, mask in [('FN', (Y[:, i] == 1) & ~flag[:, i]), ('FP', (Y[:, i] == 0) & flag[:, i])]:
        s = df[mask].sample(min(k, int(mask.sum())), random_state=42).copy()
        s['label'] = l; s['error_type'] = et; s['prob'] = Pf[s.index, i].round(3); s['true_labels'] = s[LABELS].apply(lambda r: ','.join([x for x in LABELS if r[x] == 1]), axis=1)
        out.append(s)
S = pd.concat(out)
t = S.comment_text.fillna('').str.lower()
S['auto_has_swear'] = t.str.contains(SWEAR, regex=True).astype(int)
S['auto_has_question'] = t.str.contains(r'\?').astype(int)
S['auto_has_quote'] = t.str.contains(r'["\u201c\u201d]').astype(int)
S['auto_over_100_words'] = (S.words > 100).astype(int)
TAGS = ['doubtful_label', 'no_swear_toxicity', 'rhetorical_or_sarcasm', 'metaphor', 'rare_or_obfuscated',
        'quotation_reference', 'swear_in_nontoxic', 'cut_off_128']
for c in TAGS: S[c] = ''
cols = ['id', 'label', 'error_type', 'prob', 'true_labels', 'comment_text', 'auto_has_swear', 'auto_has_question', 'auto_has_quote', 'auto_over_100_words'] + TAGS
S[cols].to_csv(f'{PROJECT_DIR}/results/tables/errors_to_tag.csv', index=False)
print(f'\nSaved {len(S)} errors to tag: results/tables/errors_to_tag.csv')
print(S.groupby(['label', 'error_type']).size())
pd.set_option('display.max_colwidth', 150)
print('\nMost confident false positives (toxic):')
print(S[(S.label == 'toxic') & (S.error_type == 'FP')].sort_values('prob', ascending=False).assign(text=lambda d: d.comment_text.str.slice(0, 200))[['prob', 'text']].head(8).to_string(index=False))
print('\nLowest-probability false negatives (toxic):')
print(S[(S.label == 'toxic') & (S.error_type == 'FN')].sort_values('prob').assign(text=lambda d: d.comment_text.str.slice(0, 200))[['prob', 'text']].head(8).to_string(index=False))
print('CELL 2 DONE')

In [ ]:
# ===== NB15 | CELL 3: summarise your tagging (run AFTER you saved errors_tagged.csv) =====
# WHAT: turn your manual tags into percentages per label and error type, and draw the chart.
# WHY:  this is the error-class table of the van Aken paper, now for OUR model.
import sys, importlib, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
TAGS = ['doubtful_label', 'no_swear_toxicity', 'rhetorical_or_sarcasm', 'metaphor', 'rare_or_obfuscated',
        'quotation_reference', 'swear_in_nontoxic', 'cut_off_128']
path = f'{PROJECT_DIR}/results/tables/errors_tagged.csv'
assert os.path.exists(path), 'Save your tagged file as results/tables/errors_tagged.csv first'
d = pd.read_csv(path)
d[TAGS] = d[TAGS].apply(pd.to_numeric, errors='coerce')
reviewed = d[TAGS].notna().any(axis=1)                      # a row counts as reviewed if you typed anything in a tag column
print(f'Reviewed rows: {int(reviewed.sum())} of {len(d)}')
d = d[reviewed].copy(); d[TAGS] = d[TAGS].fillna(0).astype(int)
rows = []
for (lab, et), g in d.groupby(['label', 'error_type']):
    r = {'label': lab, 'error_type': et, 'n_sampled': len(g)}
    r.update({t: f'{100 * g[t].mean():.0f}%' for t in TAGS}); rows.append(r)
T = pd.DataFrame(rows); print(T.to_string(index=False))
T.to_csv(f'{PROJECT_DIR}/results/tables/T9_error_classes.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, et in zip(ax, ['FN', 'FP']):
    g = d[(d.label == 'toxic') & (d.error_type == et)]
    (100 * g[TAGS].mean()).sort_values().plot(kind='barh', ax=a, color='#D9776B' if et == 'FN' else '#4C72B0')
    a.set_title(f'toxic: {"missed" if et == "FN" else "false alarms"} (n={len(g)})'); a.set_xlabel('% of sampled errors')
plt.suptitle('F16: error classes'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F16_error_classes.png', dpi=150); plt.show()
print('CELL 3 DONE')